In [1]:
import tensorflow as tf
from tensorflow import keras

In [3]:
import helper_functions

In [4]:
data_df = helper_functions.do_all_setup()

/Users/movsesyanae/Programming/Research/LLNL/alex-performance-modeling/medium-data-models/helper_functions.py:102: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raja_df.machine[raja_df['machine'] == 'ec2-c5n'] = 'ec2-c5.metal'
/Users/movsesyanae/Programming/Research/LLNL/alex-performance-modeling/medium-data-models/helper_functions.py:103: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raja_df.machine[raja_df['machine'] == 'ec2-c6i'] = 'ec2-c6i.metal'
/Users/movsesyanae/Programming/Research/LLNL/alex-performance-modeling/medium-data-models/helper_functions.py:110: FutureWarning: The frame.append method is deprecated and

In [9]:
from sklearn.model_selection import train_test_split


x_all, y_all = helper_functions.split_x_y(data_df)
x_all = x_all.to_numpy()
y_all = y_all.to_numpy()
x_train, x_test, y_train, y_test = train_test_split(x_all, y_all, test_size = 0.20, random_state = 0)

x_train = x_train.reshape((x_train.shape[0], 1, x_train.shape[1]))
x_test = x_test.reshape((x_test.shape[0], 1, x_test.shape[1]))

y_test = y_test.reshape((y_test.shape[0], 1))
y_train = y_train.reshape((y_train.shape[0], 1))


In [10]:
def create_model(num_cols):
    input_layer = keras.layers.Input(shape=(1, num_cols))
    # input_layer = keras.layers.Input(shape=(num_cols, 1))
    # activation_function = 'tanh'
    activation_function = 'relu'
    # l1 = keras.layers.Conv1D(filters=64, kernel_size=1)(input_layer)
    
    # l1 = keras.layers.Conv1D(filters=32, kernel_size=1)(l1)
    
    # l2 = keras.layers.Dense(units=1, activation=activation_function)(l1)
    
    l1 = keras.layers.Dense(units=32, activation=activation_function)(input_layer)
    l3 = keras.layers.Dense(units=8, activation=activation_function)(l1)
    dropout = tf.keras.layers.Dropout(0.2, noise_shape=None, seed=None)(l3)
    l4 = keras.layers.Dense(units=1, )(dropout)
    output_layer = keras.layers.Reshape(target_shape=(1,))(l4)
    model = keras.Model(input_layer, output_layer)
    return model
model = create_model(x_test.shape[2])
display(model.summary())

2023-11-15 16:18:29.430252: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2023-11-15 16:18:29.430280: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2023-11-15 16:18:29.430291: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2023-11-15 16:18:29.430368: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2023-11-15 16:18:29.430412: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1, 147)]          0         
                                                                 
 dense (Dense)               (None, 1, 32)             4736      
                                                                 
 dense_1 (Dense)             (None, 1, 8)              264       
                                                                 
 dropout (Dropout)           (None, 1, 8)              0         
                                                                 
 dense_2 (Dense)             (None, 1, 1)              9         
                                                                 
 reshape (Reshape)           (None, 1)                 0         
                                                                 
Total params: 5009 (19.57 KB)
Trainable params: 5009 (19.57 K

None

In [12]:
callbacks = [
    # keras.callbacks.ModelCheckpoint(
    #     "relative-reg.h5", save_best_only=True, monitor="val_loss"
    # ),
    # keras.callbacks.ReduceLROnPlateau(
    #     monitor="val_loss", factor=0.5, patience=20, min_lr=0.0001
    # ),
    # keras.callbacks.EarlyStopping(monitor="val_loss", patience=50, verbose=1),
]
model.compile(
    loss='mae',
    metrics=["accuracy", 'mape', 'mse'],
    
)

epochs = 30
batch_size = 8

history = model.fit(
    x_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=callbacks,
    validation_data=(x_test, y_test),
    verbose=1,
)

Epoch 1/30


2023-11-15 16:21:37.723454: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


805/805 [==============================] - ETA: 0s - loss: 96026.4609 - accuracy: 0.1406 - mape: 25808734.0000 - mse: 55403236818944.0000

2023-11-15 16:21:55.273438: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


805/805 [==============================] - 22s 22ms/step - loss: 96026.4609 - accuracy: 0.1406 - mape: 25808734.0000 - mse: 55403236818944.0000 - val_loss: 1073.3835 - val_accuracy: 0.3207 - val_mape: 23597746.0000 - val_mse: 260803040.0000
Epoch 2/30
805/805 [==============================] - 13s 17ms/step - loss: 877.7469 - accuracy: 0.3268 - mape: 33705788.0000 - mse: 228722720.0000 - val_loss: 1073.2655 - val_accuracy: 0.3207 - val_mape: 30484114.0000 - val_mse: 260802576.0000
Epoch 3/30
805/805 [==============================] - 13s 17ms/step - loss: 877.7281 - accuracy: 0.3268 - mape: 34981972.0000 - mse: 228722672.0000 - val_loss: 1073.2657 - val_accuracy: 0.3207 - val_mape: 30530396.0000 - val_mse: 260802576.0000
Epoch 4/30
805/805 [==============================] - 13s 17ms/step - loss: 877.7280 - accuracy: 0.3268 - mape: 34992304.0000 - mse: 228722784.0000 - val_loss: 1073.2653 - val_accuracy: 0.3207 - val_mape: 30464678.0000 - val_mse: 260802576.0000
Epoch 5/30
805/805 [====

KeyboardInterrupt: 